# Image preprocessing: precomputed encoder features

Produces the trainable image-side data for the multimodal fusion experiments -- nothing about
*how* it gets trained (fusion architecture, gating, which subset to train/evaluate on) is decided
here; that is explicitly deferred to the fusion notebooks. This notebook's only job is: given the
sample-image associations already computed in `cgm_preprocessing.ipynb`
(`image_path`/`has_image`/`time_since_meal`, already saved on every sample in
`processed_native5min/{train,val,test}.pkl`), turn every referenced photo into a fixed-dimension
feature vector, for each of three candidate frozen image encoders, so a later notebook can look
one up by key instead of re-running a CNN/ViT over raw pixels on every training epoch.

**Only samples with `has_image=True` need anything here.** The ~80% of samples with no matched
photo require no image preprocessing at all -- how they're represented at train time (a zero
vector, a gate, something else) is a modelling decision for later, not a preprocessing one.

**Deduplication matters**: because a single meal photo falls within the 60-minute lookback of many
consecutive sliding-window samples, the 22,512 `has_image=True` sample-rows across all three
splits resolve to only 2,812 *unique* photos. Every encoder therefore only has to run once per
unique photo, not once per sample-row -- an ~8x reduction in encoder compute.

**Three encoders compared from scratch** (deliberately not reusing the pilot-phase conclusion that
CLIP was best -- the sample set, resolution, and lookback window have all changed since then):
- `resnet18` (torchvision, ImageNet-pretrained, 512-d)
- `vit` (torchvision ViT-B/16, ImageNet-pretrained, 768-d)
- `clip` (`openai/clip-vit-base-patch16` vision tower, 768-d)

All three are used **frozen** (no gradient, `eval()` mode) -- consistent with the "lightweight
fusion given limited data" requirement from the multimodal architecture design
(Methodology: Model Architecture): with a few thousand images total, fine-tuning a CNN/ViT from
this dataset alone would overfit badly.

**Preprocessing recipe** (identical to the pilot phase's `preprocess/dataset.py` /
`preprocess/precompute_features.py`, reused deliberately for continuity): `SquarePad(224)` --
pad the short side to a square, then resize to 224x224, so the full photo is preserved with no
cropping -- followed by ImageNet mean/std normalization. No augmentation here: augmentation only
matters for a model that still sees raw pixels at train time, and precomputed frozen features
never do.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..')  # local: this notebook's own project root
                                    # (preprocessing -> 复现 -> Dissertation)

REPO_ROOT = PROJECT_ROOT / '复现'
DATA_ROOT = PROJECT_ROOT / 'CGMacros'  # raw participant folders + photos/ live here
PROCESSED_DIR = REPO_ROOT / 'preprocessing' / 'processed_native5min'

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={REPO_ROOT}  DATA_ROOT={DATA_ROOT}  "
      f"PROCESSED_DIR={PROCESSED_DIR}")

In [ ]:
import json
import pickle
import time

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm

DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
print(f'device: {DEVICE}')

SPLITS = ['train', 'val', 'test']
ENCODERS = ['resnet18', 'vit', 'clip']  # compared from scratch on this pipeline's own samples --
                                          # not assumed from the pilot phase's CLIP-was-best result
IMG_SIZE = 224
BATCH_SIZE = 64

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

FEATURES_DIR = PROCESSED_DIR / 'img_features'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)


class SquarePad:
    """Pad the short side to make the image square, then resize -- preserves the entire photo,
    unlike a center-crop, at the cost of some black padding on one axis."""
    def __init__(self, size: int = IMG_SIZE, fill: int = 0):
        self.size = size
        self.fill = fill

    def __call__(self, img):
        w, h = img.size
        diff = abs(w - h)
        if w < h:
            pad = (diff // 2, 0, diff - diff // 2, 0)
        else:
            pad = (0, diff // 2, 0, diff - diff // 2)
        img = TF.pad(img, pad, fill=self.fill)
        return TF.resize(img, [self.size, self.size])


IMG_TRANSFORM = transforms.Compose([
    SquarePad(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## Collect unique images

Reads the sample-image associations already computed by `cgm_preprocessing.ipynb` -- nothing about
that matching (60-minute lookback, nearest-photo selection) is redone here. Keys are
`{participant_id}/{photo filename}`, not the raw `image_path` string, so the same key format works
regardless of whether this notebook runs locally or on Colab (where `PROJECT_ROOT`, and therefore
every sample's absolute `image_path`, differs).

In [ ]:
def image_key(sample: dict) -> str:
    return f"{sample['participant_id']}/{Path(sample['image_path']).name}"


# Two genuinely different things, kept separate throughout this notebook:
#   - has_image=False samples ("no meal photo nearby"): a normal, expected data state -- nothing
#     to preprocess, never touched below.
#   - has_image=True samples whose photo turns out to be missing or unreadable on disk: a real
#     data problem. Checked explicitly HERE (by actually trying to open every file), not just left
#     for a later step to notice -- so every key that survives into `unique_images` is *guaranteed*
#     loadable, and encoder extraction below should never need to skip anything.
candidate_paths = {}   # key -> candidate absolute path, for every has_image=True row
n_rows_with_image = 0
per_split_counts = {}

for split in SPLITS:
    with open(PROCESSED_DIR / f'{split}.pkl', 'rb') as f:
        samples = pickle.load(f)
    split_n_img = 0
    for s in samples:
        if not s['has_image']:
            continue
        n_rows_with_image += 1
        split_n_img += 1
        key = image_key(s)
        if key not in candidate_paths:
            # image_path in the pkl is relative to preprocessing/ (e.g. './CGMacros/CGMacros-001/
            # photos/xxx.jpg') -- rebuilt from DATA_ROOT + participant_id + filename instead of
            # trusting that relative path directly, so this also works unchanged on Colab.
            candidate_paths[key] = DATA_ROOT / s['participant_id'] / 'photos' / Path(s['image_path']).name
    per_split_counts[split] = {'n_samples': len(samples), 'n_with_image': split_n_img}

print('per-split sample counts:')
for split, c in per_split_counts.items():
    print(f"  {split:>5}: {c['n_samples']:>6} samples, {c['n_with_image']:>5} with image "
          f"({100 * c['n_with_image'] / c['n_samples']:.1f}%)")
print(f'\ntotal has_image sample-rows (all splits): {n_rows_with_image}')
print(f'unique candidate images: {len(candidate_paths)}')

# Actually try to open every one -- not just os.path.exists(), which would miss a present-but-
# corrupted file (e.g. truncated during a Drive sync).
unique_images = {}    # key -> path, only for images that actually opened
failed_images = {}    # key -> {'path':, 'reason':} for anything that didn't

for key, path in tqdm(candidate_paths.items(), desc='verifying images open'):
    try:
        Image.open(path).convert('RGB')
        unique_images[key] = path
    except FileNotFoundError:
        failed_images[key] = {'path': str(path), 'reason': 'missing from disk'}
    except Exception as e:
        failed_images[key] = {'path': str(path), 'reason': f'{type(e).__name__}: {e}'}

print(f'\nopenable: {len(unique_images)}/{len(candidate_paths)} unique images')
if failed_images:
    fail_path = PROCESSED_DIR / 'img_features' / 'image_load_failures.json'
    fail_path.parent.mkdir(parents=True, exist_ok=True)
    with open(fail_path, 'w') as f:
        json.dump(failed_images, f, indent=2)
    print(f'FAILED to open {len(failed_images)} images -- saved details to {fail_path}')
    for k, info in list(failed_images.items())[:5]:
        print(f"  {k}: {info['reason']}")
    print('  these sample-rows are still has_image=True in the pkl, but will have no feature '
          'vector below -- effectively no-image for the image branch, and the coverage check '
          'further down reports them separately from genuinely photo-less samples.')
else:
    print('all candidate images opened successfully -- no missing/corrupted files')

### Sample output: one concrete image-bearing row

Not just counts -- an actual sample from the pickled data, and what `SquarePad(224)` visibly does
to its photo before anything goes into an encoder. Reused as `_demo_key` further down, so the
feature-extraction and coverage-check cells can show numbers for this exact same photo.

In [ ]:
import matplotlib.pyplot as plt

with open(PROCESSED_DIR / 'train.pkl', 'rb') as f:
    _demo_samples = pickle.load(f)
_demo_sample = next(s for s in _demo_samples if s['has_image'])

print('example sample dict (image-bearing row):')
for k in ['participant_id', 'timestamp', 'image_path', 'has_image', 'time_since_meal']:
    print(f'  {k}: {_demo_sample[k]}')
print(f"  cgm_sequence shape: {_demo_sample['cgm_sequence'].shape}  "
      f"dtype: {_demo_sample['cgm_sequence'].dtype}")

_demo_key = image_key(_demo_sample)
_demo_path = unique_images[_demo_key]
print(f'\nresolved image key:  {_demo_key}')
print(f'resolved disk path:  {_demo_path}')

_demo_img = Image.open(_demo_path).convert('RGB')
_demo_padded = SquarePad(IMG_SIZE)(_demo_img)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(_demo_img)
axes[0].set_title(f'original {_demo_img.size}')
axes[0].axis('off')
axes[1].imshow(_demo_padded)
axes[1].set_title(f'after SquarePad({IMG_SIZE})')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Load each encoder

Each is loaded frozen (`eval()`, `requires_grad_(False)`) -- these produce fixed feature vectors,
not something fine-tuned on this dataset.

In [ ]:
def load_encoder(name: str):
    """Returns (model, feat_dim, forward_fn). forward_fn(model, batch_tensor) -> [B, feat_dim]."""
    import torchvision.models as tv_models

    if name == 'resnet18':
        model = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Identity()
        feat_dim = 512
        forward_fn = lambda m, x: m(x)

    elif name == 'vit':
        model = tv_models.vit_b_16(weights=tv_models.ViT_B_16_Weights.IMAGENET1K_V1)
        model.heads = nn.Identity()
        feat_dim = 768
        forward_fn = lambda m, x: m(x)

    elif name == 'clip':
        from transformers import CLIPVisionModel
        model = CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch16')
        feat_dim = 768
        forward_fn = lambda m, x: m(pixel_values=x).last_hidden_state[:, 0, :]  # CLS token

    else:
        raise ValueError(f'unknown encoder: {name}')

    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model, feat_dim, forward_fn

## Extract features

One pass over the unique images in `unique_images` per encoder (not per sample-row), batched,
with a visible progress bar. Every key here already opened successfully in the previous cell, so
in the normal case `skipped` below should always come out empty -- the `try/except` is kept as a
defensive fallback (e.g. a transient read error on a Colab-mounted Drive), not because failures
are expected. If it ever *isn't* empty, that's a genuine warning printed loudly, not a silent
drop -- something changed between the open-check above and this pass.

In [ ]:
def extract_features(encoder_name: str) -> dict:
    model, feat_dim, forward_fn = load_encoder(encoder_name)
    print(f"\n{'-' * 16}encoder={encoder_name}  (feat_dim={feat_dim}){'-' * 16}")

    keys = list(unique_images.keys())
    features = {}
    skipped = []

    pbar = tqdm(range(0, len(keys), BATCH_SIZE), desc=f'{encoder_name}')
    for i in pbar:
        batch_keys = keys[i: i + BATCH_SIZE]
        imgs, valid_keys = [], []
        for k in batch_keys:
            path = unique_images[k]
            try:
                img = Image.open(path).convert('RGB')
                imgs.append(IMG_TRANSFORM(img))
                valid_keys.append(k)
            except Exception as e:
                skipped.append((k, str(e)))

        if not imgs:
            continue

        batch_tensor = torch.stack(imgs).to(DEVICE)
        with torch.no_grad():
            feats = forward_fn(model, batch_tensor).cpu().numpy()

        for k, feat in zip(valid_keys, feats):
            features[k] = feat.astype(np.float32)

        pbar.set_postfix(encoded=len(features), skipped=len(skipped))

    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    print(f'{encoder_name}: encoded {len(features)}/{len(keys)} images '
          f'({len(skipped)} skipped)')
    if skipped:
        print(f'  \u26a0\ufe0f UNEXPECTED: these {len(skipped)} keys opened fine during the '
              f'earlier verification pass but failed here -- investigate, do not ignore:')
        print('  ', skipped[:5], '...' if len(skipped) > 5 else '')
    return features, feat_dim, skipped

In [ ]:
all_skipped = {}
encoder_dims = {}

for encoder_name in ENCODERS:
    t0 = time.time()
    features, feat_dim, skipped = extract_features(encoder_name)
    elapsed = time.time() - t0
    encoder_dims[encoder_name] = feat_dim
    all_skipped[encoder_name] = skipped

    out_path = FEATURES_DIR / f'img_features_{encoder_name}.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump(features, f)
    print(f'saved {len(features)} x {feat_dim}-d features to {out_path}  '
          f'({out_path.stat().st_size / 1e6:.1f} MB, {elapsed:.0f}s)')

    # Sample output: not just "it saved a dict" -- one concrete vector from it, same demo photo
    # (_demo_key) as the SquarePad visualization above, so it's the same example throughout.
    example_key = _demo_key if _demo_key in features else next(iter(features))
    example_vec = features[example_key]
    print(f"  sample output -> key='{example_key}'")
    print(f"    shape={example_vec.shape}  dtype={example_vec.dtype}  "
          f"first5={np.round(example_vec[:5], 3)}  L2norm={np.linalg.norm(example_vec):.2f}")

## Coverage check

A three-way breakdown per encoder, not a single pass/fail count -- these are three genuinely
different situations and this notebook keeps them visibly separate rather than collapsing them
into one number:

- **no_image**: `has_image=False` -- no meal photo was ever expected nearby. Normal, the large
  majority of samples, not touched anywhere in this notebook.
- **known_failure**: `has_image=True`, but the photo failed to open in the "collect unique images"
  cell above (missing from disk or corrupted) -- a real data problem, already logged to
  `image_load_failures.json`. Expected count on this dataset: 0.
- **resolved**: `has_image=True` and successfully maps to a saved feature vector -- the normal
  success case.

Anything that doesn't fall into one of these three buckets (**unexplained**) would mean a sample
lost its feature for a reason this notebook never saw -- a real bug, not a data-quality footnote.

In [ ]:
all_features = {}

for encoder_name in ENCODERS:
    with open(FEATURES_DIR / f'img_features_{encoder_name}.pkl', 'rb') as f:
        features = pickle.load(f)
    all_features[encoder_name] = features

    n_no_image = n_known_failure = n_resolved = n_unexplained = 0
    for split in SPLITS:
        with open(PROCESSED_DIR / f'{split}.pkl', 'rb') as f:
            samples = pickle.load(f)
        for s in samples:
            if not s['has_image']:
                n_no_image += 1
                continue
            key = image_key(s)
            if key in features:
                n_resolved += 1
            elif key in failed_images:
                n_known_failure += 1
            else:
                n_unexplained += 1

    status = 'OK' if n_unexplained == 0 else 'BUG -- unexplained gap, investigate'
    print(f'{encoder_name:>9}: no_image={n_no_image}  resolved={n_resolved}  '
          f'known_failure={n_known_failure}  unexplained={n_unexplained}  [{status}]')

# Sample output: the same demo photo (_demo_key, shown above via SquarePad) resolved through all
# three encoders side by side -- concretely shows the three encoders produce genuinely different
# representations of one identical input, not just three files that happen to exist.
print(f"\nsame demo photo ('{_demo_key}') through all three encoders:")
for encoder_name, features in all_features.items():
    vec = features[_demo_key]
    print(f"  {encoder_name:>9}: shape={vec.shape}  first5={np.round(vec[:5], 3)}  "
          f"L2norm={np.linalg.norm(vec):.2f}")